In [42]:
!pip install faiss-cpu sentence-transformers openai


In [47]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.7 MB/s eta 0:00:00


In [43]:
from sentence_transformers import SentenceTransformer
import faiss
import openai

In [48]:
from groq import Groq

client = Groq(
    api_key="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"
)

In [49]:
docs=["C Joseph Vijay is a popular Indian actor and politician. He began his acting career as a child artist and later became a huge superstar in the Tamil film industry. Fans often call him Thalapathy because of his massive popularity. He has starred in numerous blockbuster movies and established himself as one of the highest paid actors in the country.",
      "Recently he made a major transition from cinema to full time politics. He officially launched his own political party named Tamilaga Vettri Kazhagam. He announced his retirement from films to completely focus on public service. This decision marked a turning point in his life and career.","His political party contested its first ever state assembly election in Tamil Nadu. The election results brought a monumental shift in the political landscape of the state. His party emerged as the single largest party by winning over one hundred seats. This victory effectively challenged decades of dominance by traditional regional parties.","Following this historic electoral victory he staked his claim to form the state government. He successfully formed a coalition government with the support of several allied political parties. He officially took the oath of office and secrecy to become the Chief Minister of Tamil Nadu. Millions of his supporters and party workers celebrated this massive achievement across the state.","As the Chief Minister he immediately started working on various welfare measures for the public. He took swift action to ensure the safety and security of citizens by launching special police initiatives. His administration is focused on fulfilling the promises made to the people during his election campaign. He operates from the state secretariat in Chennai to govern the state effectively."]

In [50]:
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(docs).astype("float32")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [51]:
doc_embeddings.shape

(5, 384)

In [52]:
dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [61]:
query ="Why vijay wants to be a CM?"
query_embedding = model.encode([query]).astype("float32")
top_k = 2
distances, indices=index.search(query_embedding,top_k)

retrieved_chunks = [docs[i] for i in indices[0]]
print("Retrieved chunks:")
for chunk in retrieved_chunks:
  print("-",chunk)



Retrieved chunks:
- C Joseph Vijay is a popular Indian actor and politician. He began his acting career as a child artist and later became a huge superstar in the Tamil film industry. Fans often call him Thalapathy because of his massive popularity. He has starred in numerous blockbuster movies and established himself as one of the highest paid actors in the country.
- As the Chief Minister he immediately started working on various welfare measures for the public. He took swift action to ensure the safety and security of citizens by launching special police initiatives. His administration is focused on fulfilling the promises made to the people during his election campaign. He operates from the state secretariat in Chennai to govern the state effectively.


In [62]:
indices

array([[0, 4]])

In [65]:
from groq import Groq

client = Groq(
    api_key="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"
)

context = "\n".join(retrieved_chunks)

prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}

Answer:
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": prompt}


    ],
    temperature=1
)

print("Final Answer:\n")
print(response.choices[0].message.content)

Final Answer:

The context does not mention why Vijay wants to be a Chief Minister (CM). It only mentions that he is the Chief Minister and describes his actions and focus after taking office.
